# Ingest Pipelines

## Connect to ElasticSearch

In [1]:
from pprint import pprint
from elasticsearch import Elasticsearch

es = Elasticsearch('http://localhost:9200')
client_info = es.info()
print('Connected to Elasticsearch!')
pprint(client_info.body)

Connected to Elasticsearch!
{'cluster_name': 'docker-cluster',
 'cluster_uuid': 'uN3swH3hQG-sv9AINTnPIg',
 'name': 'd274ca1284bc',
 'tagline': 'You Know, for Search',
 'version': {'build_date': '2024-08-05T10:05:34.233336849Z',
             'build_flavor': 'default',
             'build_hash': '1a77947f34deddb41af25e6f0ddb8e830159c179',
             'build_snapshot': False,
             'build_type': 'docker',
             'lucene_version': '9.11.1',
             'minimum_index_compatibility_version': '7.0.0',
             'minimum_wire_compatibility_version': '7.17.0',
             'number': '8.15.0'}}


## Create the pipeline

In [2]:
from pprint import pprint

response = es.ingest.put_pipeline(
    id='lowercase_pipeline',
    description='This pipeline transforms the text to lowercase',
    processors=[
        {
            "lowercase": {
                "field": "text"
            }
        }
    ]
)
pprint(response.body)

{'acknowledged': True}


## Get the pipeline

In [3]:
response = es.ingest.get_pipeline(id='lowercase_pipeline')
pprint(response.body)

{'lowercase_pipeline': {'description': 'This pipeline transforms the text to '
                                       'lowercase',
                        'processors': [{'lowercase': {'field': 'text'}}]}}


## Delete a pipeline

In [4]:
response = es.ingest.delete_pipeline(id='lowercase_pipeline')
pprint(response.body)

{'acknowledged': True}


## Simulate a pipeline

The simulate method allows you to give the pipeline fake data just to test if it is working or not. This is usually done before applying the pipeline to your real index and data.

In [5]:
response = es.ingest.put_pipeline(
    id='lowercase_pipeline',
    description='This pipeline transforms the text to lowercase',
    processors=[
        {
            "lowercase": {
                "field": "text"
            }
        }
    ]
)
pprint(response.body)

{'acknowledged': True}


Inside the docs list, we are providing some test data. After executing the cell, nothing will be indexed. You will just get back how the documents will look like after the transformation.

In [6]:
response = es.ingest.simulate(
    id='lowercase_pipeline',
    docs=[
        {
            "_index": "my_index",
            "_id": "1",
            "_source": {
                "text": "HELLO WORLD"
            }
        }
    ]
)
pprint(response.body)

{'docs': [{'doc': {'_id': '1',
                   '_index': 'my_index',
                   '_ingest': {'timestamp': '2026-09-28T19:34:07.572103042Z'},
                   '_source': {'text': 'hello world'},
                   '_version': '-3'}}]}


## Use the pipeline
Let's read the data and make the text uppercased to see if the lowercase_pipeline will be executed before indexing the documents.

In [7]:
import json

dummy_data = json.load(open("data/dummy_data.json"))
for i, document in enumerate(dummy_data):
    uppercased_text = document['text'].upper()
    document['text'] = uppercased_text
    dummy_data[i] = document

dummy_data

[{'title': 'Getting Started with Elasticsearch',
  'text': 'ELASTICSEARCH IS A DISTRIBUTED SEARCH AND ANALYTICS ENGINE BUILT ON TOP OF APACHE LUCENE. IT ALLOWS YOU TO STORE, SEARCH, AND ANALYZE LARGE VOLUMES OF DATA QUICKLY.',
  'created_on': '2026-01-05'},
 {'title': 'Understanding Inverted Indexes',
  'text': 'AN INVERTED INDEX MAPS CONTENT, SUCH AS WORDS OR NUMBERS, TO ITS LOCATIONS IN A DATABASE FILE, DOCUMENT, OR SET OF DOCUMENTS, ENABLING FAST FULL-TEXT SEARCHES.',
  'created_on': '2026-02-14'},
 {'title': 'Introduction to Vector Search',
  'text': 'VECTOR SEARCH USES EMBEDDINGS TO REPRESENT DATA AS POINTS IN A HIGH-DIMENSIONAL SPACE, ALLOWING SIMILARITY SEARCH BASED ON SEMANTIC MEANING RATHER THAN EXACT KEYWORD MATCHES.',
  'created_on': '2026-03-22'},
 {'title': 'Building REST APIs with FastAPI',
  'text': 'FASTAPI IS A MODERN, FAST WEB FRAMEWORK FOR BUILDING APIS WITH PYTHON BASED ON STANDARD TYPE HINTS, OFFERING AUTOMATIC INTERACTIVE DOCUMENTATION.',
  'created_on': '2026-05-

In [8]:
es.indices.delete(index='my_index', ignore_unavailable=True)
es.indices.create(index='my_index')

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'my_index'})

Now, we pass the lowercase_pipeline to the bulk method. It will perform the transformations before indexing the documents.

In [9]:
operations = []
for document in dummy_data:
    operations.append({'index': {'_index': 'my_index'}})
    operations.append(document)

response = es.bulk(operations=operations, pipeline='lowercase_pipeline')
pprint(response.body)

{'errors': False,
 'ingest_took': 14,
 'items': [{'index': {'_id': 'QPuD6aABh8YRbAFmuEmF',
                      '_index': 'my_index',
                      '_primary_term': 1,
                      '_seq_no': 0,
                      '_shards': {'failed': 0, 'successful': 1, 'total': 2},
                      '_version': 1,
                      'result': 'created',
                      'status': 201}},
           {'index': {'_id': 'QfuD6aABh8YRbAFmuEmG',
                      '_index': 'my_index',
                      '_primary_term': 1,
                      '_seq_no': 1,
                      '_shards': {'failed': 0, 'successful': 1, 'total': 2},
                      '_version': 1,
                      'result': 'created',
                      'status': 201}},
           {'index': {'_id': 'QvuD6aABh8YRbAFmuEmG',
                      '_index': 'my_index',
                      '_primary_term': 1,
                      '_seq_no': 2,
                      '_shards': {'failed': 0

After indexing the documents, we can see that the text field for all documents has been lowercased. This indicates the pipeline did run with no issues.

In [10]:
response = es.search(index='my_index')
hits = response.body['hits']['hits']

for hit in hits:
    print(hit['_source'])

{'title': 'Getting Started with Elasticsearch', 'created_on': '2026-01-05', 'text': 'elasticsearch is a distributed search and analytics engine built on top of apache lucene. it allows you to store, search, and analyze large volumes of data quickly.'}
{'title': 'Understanding Inverted Indexes', 'created_on': '2026-02-14', 'text': 'an inverted index maps content, such as words or numbers, to its locations in a database file, document, or set of documents, enabling fast full-text searches.'}
{'title': 'Introduction to Vector Search', 'created_on': '2026-03-22', 'text': 'vector search uses embeddings to represent data as points in a high-dimensional space, allowing similarity search based on semantic meaning rather than exact keyword matches.'}
{'title': 'Building REST APIs with FastAPI', 'created_on': '2026-05-01', 'text': 'fastapi is a modern, fast web framework for building apis with python based on standard type hints, offering automatic interactive documentation.'}
{'title': 'Data Pi

## Pipeline failure


### 1. Not handling the failure
In this scenario, we don’t handle failures with ignore_failure or on_failure. Instead, the pipeline will raise an exception, halting execution of any further processes, and the document will not be indexed.

In [11]:
response = es.ingest.put_pipeline(
    id='pipeline_1',
    description='Pipeline with multiple transformations',
    processors=[
        {
            "lowercase": {
                "field": "text",
            }
        },
        {
            "set": {
                "field": "text",
                "value": "CHANGED BY PIPELINE",
            }
        },
    ]
)
pprint(response.body)

{'acknowledged': True}


In [12]:
document = {
    'title': 'Sample Title 4',
    'created_on': '2024-09-25',
}

response = es.index(
    index='my_index',
    pipeline='pipeline_1',
    body=document
)
pprint(response.body)

BadRequestError: BadRequestError(400, 'illegal_argument_exception', 'field [text] not present as part of path [text]')

### 2. Handling the failure
To handle the failures, we use `ignore_failure` or define an `on_failure` block. With `ignore_failure`, the pipeline will skip over the failed step and continue executing subsequent processes without interrupting the flow, allowing other documents to be indexed.

Alternatively, with `on_failure`, we can specify custom error-handling steps, such as logging the error, retrying, or sending notifications, ensuring the pipeline proceeds even if one step encounters an issue.

In [13]:
response = es.ingest.put_pipeline(
    id='pipeline_2',
    description='Pipeline with multiple transformations, handling and ignoring failures',
    processors=[
        {
            "lowercase": {
                "field": "text",
                "on_failure": [
                    {
                        "set": {
                            "field": "text",
                            "value": "FAILED TO LOWERCASE",
                            "ignore_failure": True,
                        }
                    }
                ]
            }
        },
        {
            "set": {
                "field": "new_field",
                "value": "ADDED BY PIPELINE",
                "ignore_failure": True,
            }
        },
    ]
)
pprint(response.body)

{'acknowledged': True}


In [14]:
document = {
    'title': 'Sample Title 4',
    'created_on': '2024-09-25',
}

response = es.index(
    index='my_index',
    pipeline='pipeline_2',
    body=document
)
pprint(response.body)

{'_id': 'RvuH6aABh8YRbAFmWkmV',
 '_index': 'my_index',
 '_primary_term': 1,
 '_seq_no': 6,
 '_shards': {'failed': 0, 'successful': 1, 'total': 2},
 '_version': 1,
 'result': 'created'}


In [15]:
response = es.search(index='my_index')
hits = response.body['hits']['hits']
for hit in hits:
    print(hit['_source'])

{'title': 'Getting Started with Elasticsearch', 'created_on': '2026-01-05', 'text': 'elasticsearch is a distributed search and analytics engine built on top of apache lucene. it allows you to store, search, and analyze large volumes of data quickly.'}
{'title': 'Understanding Inverted Indexes', 'created_on': '2026-02-14', 'text': 'an inverted index maps content, such as words or numbers, to its locations in a database file, document, or set of documents, enabling fast full-text searches.'}
{'title': 'Introduction to Vector Search', 'created_on': '2026-03-22', 'text': 'vector search uses embeddings to represent data as points in a high-dimensional space, allowing similarity search based on semantic meaning rather than exact keyword matches.'}
{'title': 'Building REST APIs with FastAPI', 'created_on': '2026-05-01', 'text': 'fastapi is a modern, fast web framework for building apis with python based on standard type hints, offering automatic interactive documentation.'}
{'title': 'Data Pi

# Ingest Processors

## Connect to ElasticSearch

In [16]:
from pprint import pprint
from elasticsearch import Elasticsearch

es = Elasticsearch('http://localhost:9200')
client_info = es.info()
print('Connected to Elasticsearch!')
pprint(client_info.body)

Connected to Elasticsearch!
{'cluster_name': 'docker-cluster',
 'cluster_uuid': 'uN3swH3hQG-sv9AINTnPIg',
 'name': 'd274ca1284bc',
 'tagline': 'You Know, for Search',
 'version': {'build_date': '2024-08-05T10:05:34.233336849Z',
             'build_flavor': 'default',
             'build_hash': '1a77947f34deddb41af25e6f0ddb8e830159c179',
             'build_snapshot': False,
             'build_type': 'docker',
             'lucene_version': '9.11.1',
             'minimum_index_compatibility_version': '7.0.0',
             'minimum_wire_compatibility_version': '7.17.0',
             'number': '8.15.0'}}


Common ingest processors
Here’s a look at some frequently used ingest processors:

1. Convert: Changes the data type of a field.
2. Rename: Changes the name of a field.
3. Set: Assigns a specified value to a field.
4. HTML Strip: Strips HTML tags from a field's content.
5. Lowercase: Transforms the text in a field to lowercase.
6. Uppercase: Transforms the text in a field to uppercase.
7. Trim: Removes whitespace from the beginning and end of a field's value.
8. Split: Divides the field content into an array, using a comma , as the delimiter.
9. Remove: Deletes a field from the document.
10. Append: Adds a value to an array field.


## 1. Creating the document

We'll apply all these common processors to this document to demonstrate how each one works.

In [17]:
document = {
    "price": "100.50",
    "old_name": "old_value",
    "description": "<p>This is a description with HTML.</p>",
    "username": "UserNAME",
    "category": "books",
    "title": "   Example Title with Whitespace   ",
    "tags": "tag1,tag2,tag3",
    "temporary_field": "This field should be removed"
}

## 2. Creating the pipeline

In [18]:
pipeline_body = {
    "description": "Pipeline to demonstrate various ingest processors",
    "processors": [
        {
            "convert": {
                "field": "price",
                "type": "float",
                "ignore_missing": True
            }
        },
        {
            "rename": {
                "field": "old_name",
                "target_field": "new_name"
            }
        },
        {
            "set": {
                "field": "status",
                "value": "active"
            }
        },
        {
            "html_strip": {
                "field": "description"
            }
        },
        {
            "lowercase": {
                "field": "username"
            }
        },
        {
            "uppercase": {
                "field": "category"
            }
        },
        {
            "trim": {
                "field": "title"
            }
        },
        {
            "split": {
                "field": "tags",
                "separator": ","
            }
        },
        {
            "remove": {
                "field": "temporary_field"
            }
        },
        {
            "append": {
                "field": "tags",
                "value": ["new_tag"]
            }
        }
    ]
}

pipeline_id = "multi_steps_pipeline"
es.ingest.put_pipeline(id=pipeline_id, body=pipeline_body)
print(f"Pipeline '{pipeline_id}' created successfully!")

Pipeline 'multi_steps_pipeline' created successfully!


In [19]:
from pprint import pprint

es.indices.delete(index='my_index', ignore_unavailable=True)
es.indices.create(index='my_index')

response = es.index(index="my_index", document=document, pipeline=pipeline_id)
pprint(response.body)

{'_id': 'R_uY6aABh8YRbAFmJ0lX',
 '_index': 'my_index',
 '_primary_term': 1,
 '_seq_no': 0,
 '_shards': {'failed': 0, 'successful': 1, 'total': 2},
 '_version': 1,
 'result': 'created'}


In [20]:
response = es.search(index='my_index')
hits = response.body['hits']['hits']

for hit in hits:
    pprint(hit['_source'])

{'category': 'BOOKS',
 'description': '\nThis is a description with HTML.\n',
 'new_name': 'old_value',
 'price': 100.5,
 'status': 'active',
 'tags': ['tag1', 'tag2', 'tag3', 'new_tag'],
 'title': 'Example Title with Whitespace',
 'username': 'username'}
